In [10]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
covertype = fetch_ucirepo(id=31) 
  
# data (as pandas dataframes) 
X = covertype.data.features 
y = covertype.data.targets 
  
# metadata 
print(covertype.metadata) 
  
# variable information 
print(covertype.variables) 

{'uci_id': 31, 'name': 'Covertype', 'repository_url': 'https://archive.ics.uci.edu/dataset/31/covertype', 'data_url': 'https://archive.ics.uci.edu/static/public/31/data.csv', 'abstract': 'Classification of pixels into 7 forest cover types based on attributes such as elevation, aspect, slope, hillshade, soil-type, and more.', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 581012, 'num_features': 54, 'feature_types': ['Categorical', 'Integer'], 'demographics': [], 'target_col': ['Cover_Type'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1998, 'last_updated': 'Sat Mar 16 2024', 'dataset_doi': '10.24432/C50K5N', 'creators': ['Jock Blackard'], 'intro_paper': None, 'additional_info': {'summary': 'Predicting forest cover type from cartographic variables only (no remotely sensed data).  The actual forest cover type for a given observation (30 x 30 meter cell) was determined from

# SDV Models: Comprehensive Evaluation (Covertype)

This notebook contains a comprehensive evaluation of four synthetic data generation models on the **Covertype** dataset:
1. **CTGAN** - Conditional Tabular GAN
2. **CopulaGAN** - GAN-based with copula modeling
3. **Gaussian Copula** - Statistical copula-based approach
4. **TVAE** - Tabular Variational Autoencoder

Each model is evaluated using multiple fidelity metrics including KS test, JS divergence, Wasserstein distance, and more.

## Data Loading and Setup

In [11]:
covertype.data.targets.sum()

Cover_Type    1191929
dtype: int64

In [12]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Build full table (features + target) from X, y loaded above
covertype_data = pd.concat([X, y], axis=1)

# Optional subsample for faster training (full 581k rows is slow for GAN/VAE)
SAMPLE_SIZE = 50_000
if len(covertype_data) > SAMPLE_SIZE:
    covertype_data = covertype_data.sample(n=SAMPLE_SIZE, random_state=42)

# SDV metadata (same pattern as cancer notebook)
import sdv
from sdv.metadata import SingleTableMetadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(covertype_data)

print(f"Data shape: {covertype_data.shape}")
print(f"Columns: {list(covertype_data.columns)}")

Data shape: (50000, 55)
Columns: ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points', 'Wilderness_Area1', 'Soil_Type1', 'Soil_Type2', 'Soil_Type3', 'Soil_Type4', 'Soil_Type5', 'Soil_Type6', 'Soil_Type7', 'Soil_Type8', 'Soil_Type9', 'Soil_Type10', 'Soil_Type11', 'Soil_Type12', 'Soil_Type13', 'Soil_Type14', 'Soil_Type15', 'Soil_Type16', 'Soil_Type17', 'Soil_Type18', 'Soil_Type19', 'Soil_Type20', 'Soil_Type21', 'Soil_Type22', 'Soil_Type23', 'Soil_Type24', 'Soil_Type25', 'Soil_Type26', 'Soil_Type27', 'Soil_Type28', 'Soil_Type29', 'Soil_Type30', 'Soil_Type31', 'Soil_Type32', 'Soil_Type33', 'Soil_Type34', 'Soil_Type35', 'Soil_Type36', 'Soil_Type37', 'Soil_Type38', 'Soil_Type39', 'Soil_Type40', 'Wilderness_Area2', 'Wilderness_Area3', 'Wilderness_Area4', 'Cover_Type']


In [13]:
# Sample-wise count per categorical target (Cover_Type)
target_counts = y['Cover_Type'].value_counts().sort_index()
target_counts.name = 'sample_count'
print("Sample count per Cover_Type (categorical target):")
print(target_counts.to_string())
print(f"\nTotal samples: {target_counts.sum()}")

Sample count per Cover_Type (categorical target):
Cover_Type
1    211840
2    283301
3     35754
4      2747
5      9493
6     17367
7     20510

Total samples: 581012


## Utility: 10 Classifiers with Stratification

Class imbalance is handled by **stratified** train/test split and **stratified k-fold** cross-validation so each split preserves class proportions. Classifiers use `class_weight='balanced'` where supported to further mitigate imbalance.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
import numpy as np
import pandas as pd

# Target and features (use covertype_data from Data Loading; fallback to X, y)
try:
    _df = covertype_data
    target_col = 'Cover_Type'
except NameError:
    _df = pd.concat([X, y], axis=1)
    target_col = 'Cover_Type'
y_full = _df[target_col]
X_full = _df.drop(columns=[target_col])
feature_cols = [c for c in X_full.columns if c in _df.columns]
X_full = X_full[feature_cols]

# Optional: subsample for speed (stratified)
UTILITY_SAMPLE = 20_000
if len(X_full) > UTILITY_SAMPLE:
    from sklearn.model_selection import train_test_split as tts
    X_sub, _, y_sub, _ = tts(X_full, y_full, train_size=UTILITY_SAMPLE, stratify=y_full, random_state=42)
else:
    X_sub, y_sub = X_full, y_full

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_sub, y_sub, test_size=0.25, stratify=y_sub, random_state=42
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 10 classifiers (with class_weight='balanced' where supported to handle imbalance)
CLASSIFIERS = {
    'LogisticRegression': LogisticRegression(max_iter=500, class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'SVC': SVC(kernel='rbf', class_weight='balanced', random_state=42),
    'KNeighbors': KNeighborsClassifier(n_neighbors=15, weights='distance'),
    'GaussianNB': GaussianNB(),
    'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
    'MLPClassifier': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42),
}

# Stratified 5-fold CV and test evaluation
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results_utility = []
for name, clf in CLASSIFIERS.items():
    cv_scores = cross_val_score(clf, X_train_s, y_train, cv=skf, scoring='balanced_accuracy')
    clf.fit(X_train_s, y_train)
    y_pred = clf.predict(X_test_s)
    test_acc = accuracy_score(y_test, y_pred)
    test_bal = balanced_accuracy_score(y_test, y_pred)
    results_utility.append({
        'Classifier': name,
        'CV_balanced_acc_mean': cv_scores.mean(),
        'CV_balanced_acc_std': cv_scores.std(),
        'Test_accuracy': test_acc,
        'Test_balanced_accuracy': test_bal,
    })
    print(f'{name}: CV balanced_acc = {cv_scores.mean():.4f} (±{cv_scores.std():.4f}), Test balanced_acc = {test_bal:.4f}')

df_utility = pd.DataFrame(results_utility)
print('\nSummary (stratified splits + balanced_accuracy):')
print(df_utility.round(4).to_string(index=False))

## Helper Functions

Shared functions for evaluation metrics across all models.

In [14]:
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype
from scipy.spatial.distance import jensenshannon
from sklearn.preprocessing import MinMaxScaler

def _prob_vectors_numeric(real, synth, bins=30, eps=1e-12):
    """Convert numeric columns into comparable probability vectors."""
    real = pd.to_numeric(real, errors='coerce').dropna().to_numpy()
    synth = pd.to_numeric(synth, errors='coerce').dropna().to_numpy()

    edges = np.histogram_bin_edges(np.concatenate([real, synth]), bins=bins)
    r_hist, _ = np.histogram(real, bins=edges)
    s_hist, _ = np.histogram(synth, bins=edges)

    r = r_hist.astype(float) + eps
    s = s_hist.astype(float) + eps
    r /= r.sum()
    s /= s.sum()
    return r, s

def _prob_vectors_categorical(real, synth, eps=1e-12):
    """Convert categorical columns into comparable probability vectors."""
    r_counts = real.astype(str).value_counts(dropna=False)
    s_counts = synth.astype(str).value_counts(dropna=False)
    keys = r_counts.index.union(s_counts.index)
    r = r_counts.reindex(keys, fill_value=0).to_numpy(dtype=float) + eps
    s = s_counts.reindex(keys, fill_value=0).to_numpy(dtype=float) + eps
    r /= r.sum()
    s /= s.sum()
    return r, s, keys

def compute_js_divergence(real_df: pd.DataFrame,
                          synth_df: pd.DataFrame,
                          bins=30,
                          normalize=True) -> pd.DataFrame:
    """
    Compute Jensen–Shannon Divergence for each column between real and synthetic data.
    Returns a DataFrame with per-feature JS divergence values.
    """
    common_cols = [c for c in real_df.columns if c in synth_df.columns]
    real = real_df[common_cols].copy()
    synth = synth_df[common_cols].copy()

    # Optional normalization for numeric columns
    if normalize:
        num_cols = [c for c in common_cols if is_numeric_dtype(real[c])]
        scaler = MinMaxScaler()
        real[num_cols] = scaler.fit_transform(real[num_cols])
        synth[num_cols] = scaler.transform(synth[num_cols])

    results = []
    for col in common_cols:
        r_col, s_col = real[col], synth[col]

        if is_numeric_dtype(r_col):
            p, q = _prob_vectors_numeric(r_col, s_col, bins=bins)
        else:
            p, q, _ = _prob_vectors_categorical(r_col, s_col)

        # Jensen-Shannon divergence (base=2 → bounded [0,1])
        js_div = jensenshannon(p, q, base=2) ** 2

        results.append({"Feature": col, "JS_Divergence": js_div})

    return pd.DataFrame(results).sort_values("JS_Divergence")

In [15]:
from itertools import combinations
from scipy import stats
from sklearn.preprocessing import MinMaxScaler

def bivariate_quality_covertype(real_df, synth_df, target='Cover_Type'):
    """Compute bivariate quality metrics for Covertype data (multi-class target)."""
    num_cols = [c for c in real_df.columns if c != target]

    # Min-Max scaling on numeric columns
    scaler = MinMaxScaler()

    real_scaled = real_df.copy()
    synth_scaled = synth_df.copy()

    # Fit on real, transform both real and synthetic
    real_scaled[num_cols] = scaler.fit_transform(real_df[num_cols])
    synth_scaled[num_cols] = scaler.transform(synth_df[num_cols])

    results_corr = []
    results_target = []

    # Numeric-to-numeric: Δ-correlation
    for a, b in combinations(num_cols, 2):
        r_corr = real_scaled[[a, b]].corr().iloc[0, 1]
        s_corr = synth_scaled[[a, b]].corr().iloc[0, 1]
        results_corr.append({
            'var_a': a,
            'var_b': b,
            'delta_corr': abs(r_corr - s_corr)
        })

    # Numeric-to-target: Δ of average pairwise Wasserstein between classes (multi-class)
    for col in num_cols:
        r_groups = [g[col].values for _, g in real_scaled.groupby(target)]
        s_groups = [g[col].values for _, g in synth_scaled.groupby(target)]

        if len(r_groups) >= 2 and len(s_groups) >= 2:
            # Real: mean Wasserstein over all pairs of classes
            r_dists = [stats.wasserstein_distance(r_groups[i], r_groups[j])
                       for i, j in combinations(range(len(r_groups)), 2)]
            d_real = np.mean(r_dists) if r_dists else 0.0
            # Synth: same
            s_dists = [stats.wasserstein_distance(s_groups[i], s_groups[j])
                       for i, j in combinations(range(len(s_groups)), 2)]
            d_synth = np.mean(s_dists) if s_dists else 0.0
            results_target.append({
                "feature": col,
                "delta_wasserstein": abs(d_real - d_synth)
            })

    corr_df = pd.DataFrame(results_corr).sort_values("delta_corr", ascending=False)
    target_df = pd.DataFrame(results_target).sort_values("delta_wasserstein", ascending=False)

    return corr_df, target_df

## Generating synthetic data

Train all four SDV models (CTGAN, CopulaGAN, Gaussian Copula, TVAE) in one place, then run diagnostic and quality for each.

In [ ]:
from sdv.single_table import CTGANSynthesizer, CopulaGANSynthesizer, GaussianCopulaSynthesizer, TVAESynthesizer
from sdv.evaluation.single_table import evaluate_quality, run_diagnostic

# Use covertype_data and metadata from Data Loading cell; if not run yet, build them here (requires X, y from fetch)
try:
    _ = covertype_data
    _ = metadata
except NameError:
    import pandas as pd
    from sdv.metadata import SingleTableMetadata
    covertype_data = pd.concat([X, y], axis=1)
    SAMPLE_SIZE = 50_000
    if len(covertype_data) > SAMPLE_SIZE:
        covertype_data = covertype_data.sample(n=SAMPLE_SIZE, random_state=42)
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(covertype_data)
    print(f"Built covertype_data: {covertype_data.shape}")

num_rows = len(covertype_data)

# 1. CTGAN
print("Training CTGAN...")
ctgan = CTGANSynthesizer(metadata=metadata)
ctgan.fit(covertype_data)
synthetic_data_ctgan = ctgan.sample(num_rows)
diagnostic_report_ctgan = run_diagnostic(covertype_data, synthetic_data_ctgan, metadata)
quality_report_ctgan = evaluate_quality(covertype_data, synthetic_data_ctgan, metadata)
print("CTGAN done.\n")

# 2. CopulaGAN
print("Training CopulaGAN...")
copulagan = CopulaGANSynthesizer(metadata=metadata, epochs=100, verbose=True)
copulagan.fit(covertype_data)
synthetic_data_copulagan = copulagan.sample(num_rows)
diagnostic_report_copulagan = run_diagnostic(covertype_data, synthetic_data_copulagan, metadata)
quality_report_copulagan = evaluate_quality(covertype_data, synthetic_data_copulagan, metadata)
print("CopulaGAN done.\n")

# 3. Gaussian Copula
print("Training Gaussian Copula...")
gaussian_copula = GaussianCopulaSynthesizer(metadata=metadata)
gaussian_copula.fit(covertype_data)
synthetic_data_gaussian = gaussian_copula.sample(num_rows)
diagnostic_report_gaussian = run_diagnostic(covertype_data, synthetic_data_gaussian, metadata)
quality_report_gaussian = evaluate_quality(covertype_data, synthetic_data_gaussian, metadata)
print("Gaussian Copula done.\n")

# 4. TVAE
print("Training TVAE...")
tvae = TVAESynthesizer(metadata=metadata, epochs=100, verbose=True)
tvae.fit(covertype_data)
synthetic_data_tvae = tvae.sample(num_rows)
diagnostic_report_tvae = run_diagnostic(covertype_data, synthetic_data_tvae, metadata)
quality_report_tvae = evaluate_quality(covertype_data, synthetic_data_tvae, metadata)
print("TVAE done.\n")

print("All four models trained. synthetic_data_ctgan, synthetic_data_copulagan, synthetic_data_gaussian, synthetic_data_tvae")

Training CTGAN...


### 1.1 Kolmogorov-Smirnov (KS) Test

Column Shapes from SDV's Quality Report use KS Complement (1 - KS statistic) per column: higher score means real and synthetic distributions are more similar.

In [ ]:
from sdv.evaluation.single_table import QualityReport
import matplotlib.pyplot as plt
import seaborn as sns

# KS test via SDV Column Shapes (KS Complement) for all four models
models_ks = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
for name, synth in models_ks:
    report = QualityReport()
    report.generate(real_data=covertype_data, synthetic_data=synth, metadata=metadata.to_dict())
    details = report.get_details('Column Shapes')
    print(f'--- {name} ---')
    print(details)
    plt.figure(figsize=(14, 6))
    sns.barplot(x='Column', y='Score', data=details.sort_values('Score', ascending=False), palette='viridis')
    plt.title(f'{name}: Column Shapes Similarity (KS Complement)')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

### 1.2 Jensen-Shannon Divergence

Per-feature JS divergence (base 2, bounded [0,1]) between real and synthetic distributions. Lower values indicate better match.

In [ ]:
# Jensen-Shannon Divergence for all four models
models_js = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
for name, synth in models_js:
    results_js = compute_js_divergence(covertype_data, synth, bins="fd", normalize=True)
    print(f'--- {name} ---')
    print(results_js)
    results_sorted = results_js.sort_values('JS_Divergence', ascending=False)
    plt.figure(figsize=(14, 6))
    sns.barplot(x='Feature', y='JS_Divergence', data=results_sorted, palette='viridis')
    plt.xticks(rotation=90)
    plt.ylabel('JS Divergence (base 2)')
    plt.title(f'{name}: Jensen–Shannon Divergence per Feature')
    plt.tight_layout()
    plt.show()

### 1.3 Wasserstein Distance

Per-column Wasserstein (Earth Mover's) distance between real and synthetic distributions. Lower values indicate better distributional match.

In [ ]:
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import MinMaxScaler

# Wasserstein distance for all four models (per-column, after MinMax scaling)
models_w = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
for name, synth in models_w:
    real_scaled = covertype_data.copy()
    synth_scaled = synth.copy()
    scaler = MinMaxScaler()
    cols = [c for c in covertype_data.columns if c in synth.columns]
    real_scaled[cols] = scaler.fit_transform(covertype_data[cols])
    synth_scaled[cols] = scaler.transform(synth[cols])
    results_w = []
    for col in cols:
        dist = wasserstein_distance(real_scaled[col], synth_scaled[col])
        results_w.append({'Column': col, 'Wasserstein': dist})
    df_w = pd.DataFrame(results_w).sort_values('Wasserstein', ascending=False)
    print(f'--- {name} ---')
    print(df_w)
    plt.figure(figsize=(14, 6))
    sns.barplot(x='Column', y='Wasserstein', data=df_w, palette='viridis')
    plt.xticks(rotation=90)
    plt.title(f'{name}: Wasserstein Distance per Column')
    plt.tight_layout()
    plt.show()

### 1.4 Gower Distance

Gower distance handles mixed numeric and categorical columns. We report mean distance between real and synthetic samples (and mean distance from each real row to its nearest synthetic neighbor). Lower values indicate better match.

In [ ]:
import gower
import numpy as np

# Use a sample to keep the Gower matrix tractable
GOWER_SAMPLE = 1000
np.random.seed(42)
real_sample = covertype_data.sample(n=min(GOWER_SAMPLE, len(covertype_data)), random_state=42)

models_gower = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
cols_common = [c for c in real_sample.columns if c in synthetic_data_ctgan.columns]
real_sub = real_sample[cols_common]
gower_results = []

for name, synth in models_gower:
    synth_sample = synth.sample(n=min(GOWER_SAMPLE, len(synth)), random_state=42)[cols_common]
    cross_dist = gower.gower_matrix(real_sub, synth_sample)
    mean_dist = float(np.mean(cross_dist))
    mean_min_dist = float(np.mean(cross_dist.min(axis=1)))  # real to nearest synthetic
    gower_results.append({'Model': name, 'Mean Gower': mean_dist, 'Mean min (real→synth)': mean_min_dist})
    print(f'--- {name} ---')
    print(f'  Mean Gower distance (real vs synth): {mean_dist:.4f}')
    print(f'  Mean distance from real to nearest synthetic: {mean_min_dist:.4f}')

df_gower = pd.DataFrame(gower_results)
plt.figure(figsize=(8, 5))
x = np.arange(len(df_gower))
w = 0.35
plt.bar(x - w/2, df_gower['Mean Gower'], width=w, label='Mean Gower')
plt.bar(x + w/2, df_gower['Mean min (real→synth)'], width=w, label='Mean min (real→synth)')
plt.xticks(x, df_gower['Model'])
plt.ylabel('Gower distance')
plt.title('1.4 Gower Distance: Real vs Synthetic (sampled)')
plt.legend()
plt.tight_layout()
plt.show()

### 1.5 t-SNE Visualization

t-SNE projects high-dimensional real and synthetic data into 2D. Overlapping Real vs Synthetic points suggest similar local structure.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Sample for tractable t-SNE (optional: increase for better detail)
TSNE_SAMPLE = 1000
np.random.seed(42)
numeric_cols = covertype_data.select_dtypes(include=[np.number]).columns.tolist()

models_tsne = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]

for name, synth in models_tsne:
    cols = [c for c in numeric_cols if c in synth.columns]
    real_s = covertype_data[cols].sample(n=min(TSNE_SAMPLE, len(covertype_data)), random_state=42)
    synth_s = synth[cols].sample(n=min(TSNE_SAMPLE, len(synth)), random_state=43)
    scaler = StandardScaler().fit(real_s)
    real_scaled = pd.DataFrame(scaler.transform(real_s), columns=cols).assign(Type='Real')
    synth_scaled = pd.DataFrame(scaler.transform(synth_s), columns=cols).assign(Type='Synthetic')
    combined = pd.concat([real_scaled, synth_scaled], ignore_index=True)
    tsne = TSNE(n_components=2, random_state=42)
    tsne_emb = tsne.fit_transform(combined[cols])
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=tsne_emb[:, 0], y=tsne_emb[:, 1], hue=combined['Type'],
                    palette={'Real': 'blue', 'Synthetic': 'orange'}, alpha=0.6)
    plt.title(f'{name}: t-SNE (Real vs Synthetic)')
    plt.xlabel('Dimension 1')
    plt.ylabel('Dimension 2')
    plt.legend(title='Data Type')
    plt.tight_layout()
    plt.show()

### 1.6 MMD (Maximum Mean Discrepancy)

MMD measures the distance between real and synthetic distributions in a reproducing kernel Hilbert space (RBF kernel). Lower values indicate better match.

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel

def get_mmd_score(real_df, synth_df, target_col='Cover_Type', gamma=1.0, sample_size=2000):
    """Compute MMD (RBF kernel) between real and synthetic data. Uses sampling if large."""
    cols = [c for c in real_df.columns if c in synth_df.columns and c != target_col]
    real = real_df[cols].copy()
    synth = synth_df[cols].copy()
    if len(real) > sample_size:
        real = real.sample(n=sample_size, random_state=42)
    if len(synth) > sample_size:
        synth = synth.sample(n=sample_size, random_state=43)
    scaler = StandardScaler()
    r_scaled = scaler.fit_transform(real)
    s_scaled = scaler.transform(synth)
    k_xx = rbf_kernel(r_scaled, r_scaled, gamma=gamma).mean()
    k_yy = rbf_kernel(s_scaled, s_scaled, gamma=gamma).mean()
    k_xy = rbf_kernel(r_scaled, s_scaled, gamma=gamma).mean()
    mmd = k_xx + k_yy - 2 * k_xy
    return max(0.0, mmd)

models_mmd = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
mmd_results = []
for name, synth in models_mmd:
    score = get_mmd_score(covertype_data, synth, target_col='Cover_Type', gamma=1.0)
    mmd_results.append({'Model': name, 'MMD': score})
    print(f'{name} - MMD (lower is better): {score:.4f}')

df_mmd = pd.DataFrame(mmd_results)
plt.figure(figsize=(8, 5))
sns.barplot(x='Model', y='MMD', data=df_mmd, palette='viridis')
plt.title('1.6 MMD: Real vs Synthetic (RBF kernel)')
plt.ylabel('MMD')
plt.tight_layout()
plt.show()

### 1.7 Cosine Similarity

Average and maximum cosine similarity between real and synthetic samples (in standardized feature space). Higher average similarity indicates better overall match; very high max similarity may suggest overfitting.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def get_cosine_similarity(real_df, synth_df, target_col='Cover_Type', sample_size=2000):
    """Compute average and max cosine similarity between real and synthetic samples."""
    cols = [c for c in real_df.columns if c in synth_df.columns and c != target_col]
    real = real_df[cols].copy()
    synth = synth_df[cols].copy()
    if len(real) > sample_size:
        real = real.sample(n=sample_size, random_state=42)
    if len(synth) > sample_size:
        synth = synth.sample(n=sample_size, random_state=43)
    scaler = StandardScaler()
    r_scaled = scaler.fit_transform(real)
    s_scaled = scaler.transform(synth)
    sim_matrix = cosine_similarity(r_scaled, s_scaled)
    avg_sim = float(sim_matrix.mean())
    max_sim = float(sim_matrix.max())
    return avg_sim, max_sim

models_cos = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
cosine_results = []
for name, synth in models_cos:
    avg_sim, max_sim = get_cosine_similarity(covertype_data, synth, target_col='Cover_Type')
    cosine_results.append({'Model': name, 'Avg Cosine Sim': avg_sim, 'Max Cosine Sim': max_sim})
    print(f'{name}: Avg = {avg_sim:.4f}, Max = {max_sim:.4f}')

df_cos = pd.DataFrame(cosine_results)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(x='Model', y='Avg Cosine Sim', data=df_cos, palette='viridis', ax=ax[0])
ax[0].set_title('Average Cosine Similarity (higher = better match)')
sns.barplot(x='Model', y='Max Cosine Sim', data=df_cos, palette='viridis', ax=ax[1])
ax[1].set_title('Max Cosine Similarity (very high may indicate overfitting)')
plt.tight_layout()
plt.show()

### 1.9 Energy Distance

Energy distance measures the distance between two distributions (real vs synthetic) from their samples; it is zero iff the distributions are the same. We report per-column energy distance (after scaling) and a summary per model. Lower values indicate better match.

In [ ]:
from scipy.stats import energy_distance

def get_energy_distance_per_column(real_df, synth_df, target_col='Cover_Type', sample_size=2000):
    """Per-column energy distance between real and synthetic (1D). Returns DataFrame and mean."""
    cols = [c for c in real_df.columns if c in synth_df.columns and c != target_col]
    real = real_df[cols].copy()
    synth = synth_df[cols].copy()
    if len(real) > sample_size:
        real = real.sample(n=sample_size, random_state=42)
    if len(synth) > sample_size:
        synth = synth.sample(n=sample_size, random_state=43)
    scaler = StandardScaler()
    real_scaled = scaler.fit_transform(real)
    synth_scaled = scaler.transform(synth)
    results = []
    for i, col in enumerate(cols):
        ed = energy_distance(real_scaled[:, i], synth_scaled[:, i])
        results.append({'Column': col, 'Energy_Distance': ed})
    df = pd.DataFrame(results).sort_values('Energy_Distance', ascending=False)
    return df, float(df['Energy_Distance'].mean())

models_ed = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
ed_summary = []
for name, synth in models_ed:
    df_ed, mean_ed = get_energy_distance_per_column(covertype_data, synth, target_col='Cover_Type')
    ed_summary.append({'Model': name, 'Mean Energy Distance': mean_ed})
    print(f'{name}: mean energy distance = {mean_ed:.4f}')
    print(df_ed.head(10).to_string(index=False))
    plt.figure(figsize=(14, 6))
    sns.barplot(x='Column', y='Energy_Distance', data=df_ed.head(20), palette='viridis')
    plt.xticks(rotation=90)
    plt.title(f'{name}: Per-column Energy Distance (top 20)')
    plt.tight_layout()
    plt.show()

df_ed_summary = pd.DataFrame(ed_summary)
plt.figure(figsize=(8, 5))
sns.barplot(x='Model', y='Mean Energy Distance', data=df_ed_summary, palette='viridis')
plt.title('1.9 Energy Distance: Mean per Model (lower = better)')
plt.tight_layout()
plt.show()

### 1.10 Bhattacharyya Distance

Bhattacharyya distance between two distributions: D_B = -ln(BC), where BC = Σ√(p·q) over histogram bins. Zero when distributions match; larger when they differ. Reported per column (histogram-based) for all four models. Lower is better.

In [ ]:
def get_bhattacharyya_per_column(real_df, synth_df, target_col='Cover_Type', bins='fd'):
    """Per-column Bhattacharyya distance via histogram probability vectors. BC = sum(sqrt(p*q)), D_B = -ln(BC)."""
    common_cols = [c for c in real_df.columns if c in synth_df.columns and c != target_col]
    real = real_df[common_cols].copy()
    synth = synth_df[common_cols].copy()
    results = []
    for col in common_cols:
        r_col, s_col = real[col], synth[col]
        if pd.api.types.is_numeric_dtype(r_col):
            p, q = _prob_vectors_numeric(r_col, s_col, bins=bins)
        else:
            p, q, _ = _prob_vectors_categorical(r_col, s_col)
        bc = np.sum(np.sqrt(p * q))
        bc = np.clip(bc, 1e-12, 1.0)
        d_b = -np.log(bc)
        results.append({'Column': col, 'Bhattacharyya': d_b})
    df = pd.DataFrame(results).sort_values('Bhattacharyya', ascending=False)
    return df, float(df['Bhattacharyya'].mean())

models_bhat = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
bhat_summary = []
for name, synth in models_bhat:
    df_bhat, mean_bhat = get_bhattacharyya_per_column(covertype_data, synth, target_col='Cover_Type', bins='fd')
    bhat_summary.append({'Model': name, 'Mean Bhattacharyya': mean_bhat})
    print(f'{name}: mean Bhattacharyya distance = {mean_bhat:.4f}')
    print(df_bhat.head(10).to_string(index=False))
    plt.figure(figsize=(14, 6))
    sns.barplot(x='Column', y='Bhattacharyya', data=df_bhat.head(20), palette='viridis')
    plt.xticks(rotation=90)
    plt.title(f'{name}: Per-column Bhattacharyya Distance (top 20)')
    plt.tight_layout()
    plt.show()

df_bhat_summary = pd.DataFrame(bhat_summary)
plt.figure(figsize=(8, 5))
sns.barplot(x='Model', y='Mean Bhattacharyya', data=df_bhat_summary, palette='viridis')
plt.title('1.10 Bhattacharyya Distance: Mean per Model (lower = better)')
plt.tight_layout()
plt.show()

### 1.11 Manhattan Distance

Average L1 (Manhattan) distance from each synthetic sample to its nearest real neighbor in standardized feature space. Complements the L2 nearest-neighbor metric (1.8); lower values indicate synthetic points are closer to the real distribution.

In [ ]:
from sklearn.neighbors import NearestNeighbors

def get_manhattan_nn_distance(real_df, synth_df, target_col='Cover_Type', sample_size=2000):
    """Average Manhattan (L1) distance from each synthetic sample to its nearest real neighbor."""
    cols = [c for c in real_df.columns if c in synth_df.columns and c != target_col]
    real = real_df[cols].copy()
    synth = synth_df[cols].copy()
    if len(real) > sample_size:
        real = real.sample(n=sample_size, random_state=42)
    if len(synth) > sample_size:
        synth = synth.sample(n=sample_size, random_state=43)
    scaler = StandardScaler()
    real_scaled = scaler.fit_transform(real)
    synth_scaled = scaler.transform(synth)
    nn = NearestNeighbors(n_neighbors=1, metric='manhattan').fit(real_scaled)
    distances, _ = nn.kneighbors(synth_scaled)
    return float(distances.mean())

models_manhattan = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
manhattan_results = []
for name, synth in models_manhattan:
    avg_l1 = get_manhattan_nn_distance(covertype_data, synth, target_col='Cover_Type')
    manhattan_results.append({'Model': name, 'Avg Manhattan NN': avg_l1})
    print(f'{name}: Avg Manhattan distance (synth→nearest real) = {avg_l1:.4f}')

df_manhattan = pd.DataFrame(manhattan_results)
plt.figure(figsize=(8, 5))
sns.barplot(x='Model', y='Avg Manhattan NN', data=df_manhattan, palette='viridis')
plt.title('1.11 Manhattan Distance: Avg NN (Synthetic → Real, L1)')
plt.ylabel('Avg Manhattan NN (lower = closer to real)')
plt.tight_layout()
plt.show()

### 1.12 Cosine Distance

Cosine distance = 1 − cosine similarity (angle-based distance between samples in feature space). We report average and minimum cosine distance between real and synthetic samples. Lower values indicate better alignment; higher min distance suggests no synthetic point is overly close to a real one.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def get_cosine_distance(real_df, synth_df, target_col='Cover_Type', sample_size=2000):
    """Average and min cosine distance (1 - cosine_similarity) between real and synthetic samples."""
    cols = [c for c in real_df.columns if c in synth_df.columns and c != target_col]
    real = real_df[cols].copy()
    synth = synth_df[cols].copy()
    if len(real) > sample_size:
        real = real.sample(n=sample_size, random_state=42)
    if len(synth) > sample_size:
        synth = synth.sample(n=sample_size, random_state=43)
    scaler = StandardScaler()
    r_scaled = scaler.fit_transform(real)
    s_scaled = scaler.transform(synth)
    sim_matrix = cosine_similarity(r_scaled, s_scaled)
    # Cosine distance = 1 - similarity (in [0, 2] for real-valued; often clamped to [0, 1])
    dist_matrix = 1 - sim_matrix
    avg_dist = float(dist_matrix.mean())
    min_dist = float(dist_matrix.min())  # 1 - max similarity
    return avg_dist, min_dist

models_cos_dist = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
cos_dist_results = []
for name, synth in models_cos_dist:
    avg_d, min_d = get_cosine_distance(covertype_data, synth, target_col='Cover_Type')
    cos_dist_results.append({'Model': name, 'Avg Cosine Dist': avg_d, 'Min Cosine Dist': min_d})
    print(f'{name}: Avg cosine distance = {avg_d:.4f}, Min = {min_d:.4f}')

df_cos_dist = pd.DataFrame(cos_dist_results)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(x='Model', y='Avg Cosine Dist', data=df_cos_dist, palette='viridis', ax=ax[0])
ax[0].set_title('Average Cosine Distance (lower = better match)')
sns.barplot(x='Model', y='Min Cosine Dist', data=df_cos_dist, palette='viridis', ax=ax[1])
ax[1].set_title('Min Cosine Distance (1 − max similarity)')
plt.tight_layout()
plt.show()

### 1.13 Hungarian Mapping (Optimal Assignment)

Optimal one-to-one assignment between real and synthetic samples using the **Hungarian algorithm** to minimize total cost. We report the **mean distance under optimal matching** for four distance metrics: **Euclidean**, **Mahalanobis**, **Manhattan**, and **Cosine**. Lower values indicate better real–synthetic alignment. Uses a sample of points to keep the cost matrix tractable.

In [ ]:
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.metrics.pairwise import euclidean_distances, manhattan_distances, cosine_distances
import numpy as np

def hungarian_mapping_cost(real_2d, synth_2d, metric='euclidean', cov_vi=None):
    """
    Cost matrix between real (rows) and synth (cols), then optimal assignment.
    Returns mean cost per pair (total / n).
    """
    n = real_2d.shape[0]
    if metric == 'euclidean':
        C = euclidean_distances(real_2d, synth_2d)
    elif metric == 'manhattan':
        C = manhattan_distances(real_2d, synth_2d)
    elif metric == 'cosine':
        C = cosine_distances(real_2d, synth_2d)
    elif metric == 'mahalanobis':
        if cov_vi is None:
            cov = np.cov(real_2d.T)
            cov = cov + 1e-6 * np.eye(cov.shape[0])
            cov_vi = np.linalg.inv(cov)
        C = cdist(real_2d, synth_2d, metric='mahalanobis', VI=cov_vi)
    else:
        raise ValueError(metric)
    row_ind, col_ind = linear_sum_assignment(C)
    total = C[row_ind, col_ind].sum()
    return total / n

# Sample size for assignment (N×N matrix; keep moderate for speed)
HUNGARIAN_N = 500
np.random.seed(42)
cols = [c for c in covertype_data.columns if c in synthetic_data_ctgan.columns and c != 'Cover_Type']
real_sub = covertype_data[cols].sample(n=min(HUNGARIAN_N, len(covertype_data)), random_state=42)
scaler = StandardScaler()
real_scaled = scaler.fit_transform(real_sub)
# Mahalanobis: inverse covariance from real data
cov = np.cov(real_scaled.T) + 1e-6 * np.eye(real_scaled.shape[1])
cov_vi = np.linalg.inv(cov)

models_h = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
metrics_h = ['euclidean', 'mahalanobis', 'manhattan', 'cosine']

results_h = []
for name, synth in models_h:
    synth_sub = synth[cols].sample(n=min(HUNGARIAN_N, len(synth)), random_state=43)
    n = min(len(real_sub), len(synth_sub))
    R = real_scaled[:n]
    S = scaler.transform(synth_sub.iloc[:n])
    for metric in metrics_h:
        mean_cost = hungarian_mapping_cost(
            R, S,
            metric=metric,
            cov_vi=cov_vi if metric == 'mahalanobis' else None
        )
        results_h.append({'Model': name, 'Distance': metric.capitalize(), 'Hungarian_mean_cost': mean_cost})
        print(f'{name} / {metric}: mean optimal cost = {mean_cost:.4f}')

df_h = pd.DataFrame(results_h)
pivot_h = df_h.pivot(index='Model', columns='Distance', values='Hungarian_mean_cost')
print('\nHungarian mapping — mean cost per metric:')
print(pivot_h.round(4).to_string())

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for idx, metric in enumerate(metrics_h):
    ax = axes.flat[idx]
    sub = df_h[df_h['Distance'] == metric.capitalize()]
    sns.barplot(x='Model', y='Hungarian_mean_cost', data=sub, palette='viridis', ax=ax)
    ax.set_title(f'Hungarian: {metric.capitalize()} (lower = better)')
    ax.set_ylabel('Mean optimal cost')
plt.tight_layout()
plt.show()

### 1.8 Average Nearest Neighbor Distance

Average distance from each synthetic sample to its nearest real neighbor (in standardized feature space). Lower values indicate synthetic points lie closer to the real data distribution.

In [ ]:
from sklearn.neighbors import NearestNeighbors

def get_nearest_neighbor_distance(real_df, synth_df, target_col='Cover_Type', sample_size=2000):
    """Average distance from each synthetic sample to its nearest real neighbor."""
    cols = [c for c in real_df.columns if c in synth_df.columns and c != target_col]
    real = real_df[cols].copy()
    synth = synth_df[cols].copy()
    if len(real) > sample_size:
        real = real.sample(n=sample_size, random_state=42)
    if len(synth) > sample_size:
        synth = synth.sample(n=sample_size, random_state=43)
    scaler = StandardScaler()
    real_scaled = scaler.fit_transform(real)
    synth_scaled = scaler.transform(synth)
    nn = NearestNeighbors(n_neighbors=1).fit(real_scaled)
    distances, _ = nn.kneighbors(synth_scaled)
    return float(distances.mean())

models_nn = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
nn_results = []
for name, synth in models_nn:
    avg_nn = get_nearest_neighbor_distance(covertype_data, synth, target_col='Cover_Type')
    nn_results.append({'Model': name, 'Avg NN distance': avg_nn})
    print(f'{name}: Avg nearest neighbor distance (synth→real) = {avg_nn:.4f}')

df_nn = pd.DataFrame(nn_results)
plt.figure(figsize=(8, 5))
sns.barplot(x='Model', y='Avg NN distance', data=df_nn, palette='viridis')
plt.title('1.8 Average Nearest Neighbor Distance (Synthetic → Real)')
plt.ylabel('Avg NN distance (lower = closer to real)')
plt.tight_layout()
plt.show()

## 2. Bivariate Analysis

Compares pairwise structure between real and synthetic data: **Δ-correlation** (numeric–numeric) and **Δ-Wasserstein** (feature vs target, multi-class). Uses the helper `bivariate_quality_covertype` for all four models.

In [ ]:
# Bivariate quality for all four models
models_bivar = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
TOP_K = 10  # show top K pairs/features

for name, synth in models_bivar:
    corr_df, target_df = bivariate_quality_covertype(covertype_data, synth, target='Cover_Type')
    print(f'\n=== {name} ===')
    print('Top pairwise Δ-correlation (real vs synth):')
    print(corr_df.head(TOP_K).to_string(index=False))
    print('\nTop feature Δ-Wasserstein vs Cover_Type:')
    print(target_df.head(TOP_K).to_string(index=False))
    # Plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    top_corr = corr_df.head(TOP_K).copy()
    top_corr['pair'] = top_corr['var_a'] + ' / ' + top_corr['var_b']
    sns.barplot(data=top_corr, x='delta_corr', y='pair', palette='viridis', ax=axes[0])
    axes[0].set_xlabel('|Δ correlation|')
    axes[0].set_title(f'{name}: Top pairwise Δ-correlation')
    sns.barplot(data=target_df.head(TOP_K), x='delta_wasserstein', y='feature', palette='viridis', ax=axes[1])
    axes[1].set_xlabel('Δ Wasserstein (vs Cover_Type)')
    axes[1].set_title(f'{name}: Top feature–target Δ-Wasserstein')
    plt.tight_layout()
    plt.show()

## 3. Multivariate Analysis

Evaluates joint structure across many variables: **Column Pair Trends** (SDV), **correlation matrix** similarity (Frobenius norm of difference), and **PCA** projection of real vs synthetic. Run for all four models.

In [ ]:
from sklearn.decomposition import PCA
from sdv.evaluation.single_table import QualityReport

# Sample size for correlation and PCA (optional, for speed)
MULTI_SAMPLE = 2000
np.random.seed(42)
models_multi = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
numeric_cols = [c for c in covertype_data.select_dtypes(include=[np.number]).columns if c != 'Cover_Type']

for name, synth in models_multi:
    cols = [c for c in numeric_cols if c in synth.columns]
    real_sub = covertype_data[cols].sample(n=min(MULTI_SAMPLE, len(covertype_data)), random_state=42)
    synth_sub = synth[cols].sample(n=min(MULTI_SAMPLE, len(synth)), random_state=43)
    scaler = StandardScaler()
    X_real = scaler.fit_transform(real_sub)
    X_synth = scaler.transform(synth_sub)

    # 1) Column Pair Trends (SDV)
    report = QualityReport()
    report.generate(real_data=covertype_data, synthetic_data=synth, metadata=metadata.to_dict())
    pair_details = report.get_details('Column Pair Trends')
    overall_score = report.get_score()
    print(f'\n=== {name} ===')
    print(f'Overall quality score: {overall_score:.4f}')
    if len(pair_details) > 0:
        if 'Score' in pair_details.columns:
            print(f'Column Pair Trends (mean): {pair_details["Score"].mean():.4f}')
        print('Sample pair trends (first 5 rows):')
        print(pair_details.head(5).to_string(index=False))

    # 2) Correlation matrix Frobenius difference
    R_real = np.corrcoef(real_sub.T)
    R_synth = np.corrcoef(synth_sub.T)
    frob_diff = np.sqrt(np.sum((R_real - R_synth) ** 2))
    print(f'Correlation matrix Frobenius diff (lower = better): {frob_diff:.4f}')

    # 3) PCA: Real vs Synthetic
    X_all = np.vstack([X_real, X_synth])
    pca = PCA(n_components=2).fit(X_all)
    X_pca = pca.transform(X_all)
    n_real = len(X_real)
    plt.figure(figsize=(8, 6))
    plt.scatter(X_pca[:n_real, 0], X_pca[:n_real, 1], c='blue', alpha=0.4, label='Real', s=20)
    plt.scatter(X_pca[n_real:, 0], X_pca[n_real:, 1], c='orange', alpha=0.4, label='Synthetic', s=20)
    plt.title(f'{name}: PCA — Real vs Synthetic')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.legend()
    plt.tight_layout()
    plt.show()

## TRTR / TSTR Evaluation (Train on Real / Train on Synthetic, Test on Real)

- **TRTR:** Train on real data, test on real holdout (stratified split). Baseline performance.
- **TSTR:** Train on synthetic data, test on the same real holdout. Measures how well synthetic data preserves utility for downstream classification.
Both use the same 10 classifiers and stratified splits; metrics reported are accuracy and balanced accuracy.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
import numpy as np
import pandas as pd

target_col = 'Cover_Type'
feature_cols = [c for c in covertype_data.columns if c != target_col]
X_real = covertype_data[feature_cols]
y_real = covertype_data[target_col]

# Stratified split: same real test set for TRTR and TSTR
X_real_train, X_real_test, y_real_train, y_real_test = train_test_split(
    X_real, y_real, test_size=0.25, stratify=y_real, random_state=42
)
scaler = StandardScaler()
X_real_train_s = scaler.fit_transform(X_real_train)
X_real_test_s = scaler.transform(X_real_test)

CLASSIFIERS = {
    'LogisticRegression': LogisticRegression(max_iter=500, class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'SVC': SVC(kernel='rbf', class_weight='balanced', random_state=42),
    'KNeighbors': KNeighborsClassifier(n_neighbors=15, weights='distance'),
    'GaussianNB': GaussianNB(),
    'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
    'MLPClassifier': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42),
}

# TRTR: Train on Real, Test on Real
print('=' * 60)
print('TRTR — Train on Real, Test on Real')
print('=' * 60)
trtr_results = []
for name, clf in CLASSIFIERS.items():
    clf.fit(X_real_train_s, y_real_train)
    y_pred = clf.predict(X_real_test_s)
    trtr_results.append({
        'Classifier': name,
        'TRTR_accuracy': accuracy_score(y_real_test, y_pred),
        'TRTR_balanced_accuracy': balanced_accuracy_score(y_real_test, y_pred),
    })
df_trtr = pd.DataFrame(trtr_results)
print(df_trtr.round(4).to_string(index=False))

# TSTR: Train on Synthetic, Test on Real (for each of the four synthetic datasets)
synth_models = [
    ('CTGAN', synthetic_data_ctgan),
    ('CopulaGAN', synthetic_data_copulagan),
    ('Gaussian Copula', synthetic_data_gaussian),
    ('TVAE', synthetic_data_tvae),
]
tstr_results = []
for synth_name, synth_df in synth_models:
    X_synth = synth_df[feature_cols]
    y_synth = synth_df[target_col]
    X_synth_s = scaler.transform(X_synth)  # same scaler fit on real train
    for clf_name, clf in CLASSIFIERS.items():
        clf.fit(X_synth_s, y_synth)
        y_pred = clf.predict(X_real_test_s)
        tstr_results.append({
            'SynthModel': synth_name,
            'Classifier': clf_name,
            'TSTR_accuracy': accuracy_score(y_real_test, y_pred),
            'TSTR_balanced_accuracy': balanced_accuracy_score(y_real_test, y_pred),
        })

df_tstr = pd.DataFrame(tstr_results)
print('\n' + '=' * 60)
print('TSTR — Train on Synthetic, Test on Real (by synthetic model)')
print('=' * 60)
for synth_name in [m[0] for m in synth_models]:
    sub = df_tstr[df_tstr['SynthModel'] == synth_name][['Classifier', 'TSTR_accuracy', 'TSTR_balanced_accuracy']]
    print(f'\n--- {synth_name} ---')
    print(sub.round(4).to_string(index=False))

# Summary: TRTR vs TSTR (mean over classifiers per synth model)
print('\n' + '=' * 60)
print('Summary: Mean balanced accuracy (TRTR vs TSTR per synthetic model)')
print('=' * 60)
trtr_bal = df_trtr['TRTR_balanced_accuracy'].mean()
print(f'TRTR (train=real): mean balanced_acc = {trtr_bal:.4f}')
for synth_name in [m[0] for m in synth_models]:
    tstr_bal = df_tstr[df_tstr['SynthModel'] == synth_name]['TSTR_balanced_accuracy'].mean()
    print(f'TSTR (train={synth_name}): mean balanced_acc = {tstr_bal:.4f}')